In [75]:
import re
import enum
import pyranges as pr
import pandas as pd


class CIGARstep(enum.Enum):
    TARGET = 0
    QUERY = 1
    BOTH = 2

class CIGARwalker:
    __slots__ = (
        "cigar", "cigar_ops", "move_ops", "orientation",
        "tstart", "tend", "qstart", "qend",
        "target_regions", "query_regions",
        "row_idx",
        "last_start_cigar", "last_start_target", "last_start_query"
    )

    def __init__(self, tstart, tend, qstart, qend, orientation, cigar, row_idx):

        self.cigar_ops = re.compile("[0-9]+(\\=|X|D|I|M)")
        self.cigar = cigar
        self.move_ops = {
            "=": CIGARstep.BOTH,
            "M": CIGARstep.BOTH,
            "X": CIGARstep.BOTH,
            "D": CIGARstep.TARGET,
            "I": CIGARstep.QUERY
        }
        self.orientation = orientation
        assert self.orientation in [1,-1]
        self.tstart = tstart
        self.tend = tend
        if self.orientation < 0:
            self.qstart = qend * -1
            self.qend = qstart * -1
        else:
            self.qstart = qstart
            self.qend = qend
        assert self.qstart < self.qend
        self.row_idx = row_idx

        # saving state for quicker iterations
        self.last_start_cigar = 0
        self.last_start_target = self.tstart
        self.last_start_query = self.qstart
        
        return None

    def walk_cigar(self):

        for cigar_op in self.cigar_ops.finditer(self.cigar[self.last_start_cigar:]):
            cigar_start, _ = cigar_op.span()
            tmp = cigar_op.group(0)
            move_op = self.move_ops[tmp[-1]]
            step = int(tmp[:-1])
            yield cigar_start, step, move_op
        return

    def find_range(self, find_start, find_end):

        iter_t = self.last_start_target
        iter_q = self.last_start_query
        lifted_start = None
        lifted_end = None
        for cigar_start, step, move in self.walk_cigar():
            if lifted_start is None and iter_t >= find_start:
                lifted_start = iter_q
                self.last_start_cigar = cigar_start
                self.last_start_target = iter_t
                self.last_start_query = iter_q
            if move == CIGARstep.BOTH:
                iter_t += step
                iter_q += step
            elif move == CIGARstep.TARGET:
                iter_t += step
            else:
                iter_q += step
            if iter_t >= find_end:
                lifted_end = iter_q
                break
        if lifted_start is None:
            # this can happen if a small region shall be lifted over
            # and a large CIGAR step walks creates a full
            # containment for that region in a single step

            # unroll
            if move == CIGARstep.BOTH:
                iter_t -= step
                iter_q -= step
            elif move == CIGARstep.TARGET:
                # this shoudl imply that the region
                # does not exist in the query
                iter_t -= step
            else:
                # this cannot have happened
                raise
            print(iter_t)
            print(step)
            print(find_start)
            print(find_end)
            raise
        assert lifted_start is not None, f"{iter_t} - {find_start} / {find_end}"
        assert lifted_end is not None, f"{iter_t} - {find_end}"
        return lifted_start, lifted_end

paf_file = "/home/ebertp/work/projects/chrom-y-extended/wf-data/seqann/labeled_ref/HG02258.aa2a60f6.hg38-repeat-details.label-ref-aln.norm-paf.tsv.gz"
region_ann = "/home/ebertp/work/projects/chrom-y-extended/wf-data/seqann/labeled_ref/GRCh38-Y_regions_repeat-details.bed"

paf = pd.read_csv(paf_file, sep="\t")
assert paf.target_name.nunique() == 1
regions = pd.read_csv(region_ann, header=0, sep="\t")
regions.rename({"#chrom": "chrom"}, axis=1, inplace=True)

paf["target_name"] = paf["target_name"].replace({paf.target_name.iloc[0]: regions.chrom.iloc[0]}, inplace=False)

region_labels = pr.from_dict(
    {
        "Chromosome": regions.chrom,
        "Start": regions.start,
        "End": regions.end,
        "Name": regions.name,
        "region_idx": regions.index.values
    }
)

aligned_blocks = pr.from_dict(
    {
        "Chromosome": paf.target_name,
        "Start": paf.target_start,
        "End": paf.target_end,
        "align_idx": paf.index.values
    }
)

broken_alignments = aligned_blocks.intersect(region_labels)
broken_alignments = broken_alignments.join(region_labels, suffix="_region_label")

row_select = []
# why this:
# because pyranges.intersect is returning all
# intersections (how=None parameter),
# that is, overlapping regions in the source BED
# file / region annotation create various intersections.
# Select an intersect only if it does not extend beyond
# the original source region size.

broken_alignments = broken_alignments.df
priority_matches = set()
for row in broken_alignments.itertuples():
    equal_start = row.Start == row.Start_region_label
    smaller_equal_end = row.End <= row.End_region_label
    larger_start = row.Start > row.Start_region_label
    if equal_start & smaller_equal_end:
        # perfect match, potantially cut off because alignment ends
        row_select.append(row.Index)
        priority_matches.add((row.align_idx, row.region_idx))
        continue
    elif larger_start & smaller_equal_end:
        # could be a secondary containment, select only
        # if no priority match exists for this combination
        key = row.align_idx, row.region_idx
        if key in priority_matches:
            continue
        
        row_select.append(row.Index)
        continue
    else:
        # missed special case - ?
        continue

broken_alignments = broken_alignments.loc[row_select, :]

last_alignment = None

for row in broken_alignments.itertuples():
    alignment = paf.loc[row.align_idx]
    if last_alignment is None or last_alignment != row.align_idx:
        last_alignment = row.align_idx
        cw = CIGARwalker(
            alignment.target_start,
            alignment.target_end,
            alignment.query_start,
            alignment.query_end,
            alignment.align_orient,
            alignment.cg_cigar,
            row.align_idx
        )
    print(row)
    lifted_label_coord = cw.find_range(row.Start, row.End)
    print(lifted_label_coord)
    if last_alignment != 0:
        break

raise



paf["pct_matching"] = round(paf["align_matching"] / paf["query_length"] * 100, 2)

def get_max_identity_block(cigar_string):

    max_block = 0
    for block in re.finditer("[0-9]+\=", cigar_string):
        max_block = max(max_block, int(block.group(0)[:-1]))
    return max_block


paf["max_id_block"] = paf["cg_cigar"].apply(get_max_identity_block)

paf["rank_matching"] = paf["pct_matching"].rank(method="average", ascending=True, pct=True)
paf["rank_block"] = paf["max_id_block"].rank(method="average", ascending=True, pct=True)
paf["mean_rank"] = ((paf.rank_matching + paf.rank_block) / 2 * 1000).round(0).astype(int)
paf["strand"] = paf.align_orient.replace({1: "+", -1: "-"}).astype(str)

target_iv = pr.from_dict(
    {
        "Chromosome": paf.target_name,
        "Start": paf.target_start,
        "End": paf.target_end,
        "Strand": paf.strand,
        "Name": paf.query_name,
        "pd_idx": paf.index.values
    }
)

target_iv = target_iv.cluster(strand="same", by="Name")
cluster_ids = pd.Series(
    target_iv.Cluster.values,
    index=target_iv.pd_idx.values,
    name="cluster_id"
)

# build in a sanity check given the dev stage of PyRanges
_n_rows = paf.shape[0]

paf = paf.merge(cluster_ids, left_index=True, right_index=True)

assert paf.shape[0] == _n_rows

bed_rows = []
for cluster_id, alignments in paf.groupby("cluster_id"):
    assert alignments.query_name.nunique() == 1
    seq = alignments.target_name.iloc[0]
    name = alignments.query_name.iloc[0]
    strand = alignments.strand.iloc[0]
    start = alignments.target_start.min()
    end = alignments.target_end.max()
    assert start < end
    score = alignments.mean_rank.max()
    max_pct = alignments.pct_matching.max()
    max_block = alignments.max_id_block.max()
    new_label = f"{name}[IDPCT:{max_pct}|IDBLK:{max_block}]"
    bed_rows.append(
        (seq, start, end, new_label, score, strand, name, max_pct, max_block, alignments.shape[0])
    )

bed_df = pd.DataFrame.from_records(
    bed_rows,
    columns=[
        "chrom", "start", "end", "name", "score", "strand",
        "label", "max_pct_align_match", "longest_id_block",
        "merged_alignments"
    ],
)

bed_df.sort_values(["chrom", "start", "end"], inplace=True)

print(bed_df.head(10))

Pandas(Index=0, Chromosome='chrY', Start=2133016, End=2781480, align_idx=0, Start_region_label=10000, End_region_label=2781480, Name='01n_PAR1', region_idx=0)
(np.int64(1763670), np.int64(2418095))
Pandas(Index=1, Chromosome='chrY', Start=2781480, End=3049683, align_idx=0, Start_region_label=2781480, End_region_label=3049683, Name='02n_XDR1', region_idx=1)
(np.int64(2418095), np.int64(2702805))
Pandas(Index=2, Chromosome='chrY', Start=3049683, End=6234810, align_idx=0, Start_region_label=3049683, End_region_label=6234810, Name='03n_XTR1', region_idx=2)
(np.int64(2702805), np.int64(5878882))
Pandas(Index=3, Chromosome='chrY', Start=6234810, End=6248951, align_idx=0, Start_region_label=6234810, End_region_label=6532528, Name='04u_AMPL1_IR3d', region_idx=3)
(np.int64(5880709), np.int64(5911111))
Pandas(Index=6, Chromosome='chrY', Start=6245524, End=6248951, align_idx=0, Start_region_label=6245524, End_region_label=6266829, Name='05s_TSPY-small', region_idx=4)
6253668
15643
6245524
6248951

RuntimeError: No active exception to reraise